# 🏛️ Landmark Detection - Model Architecture
## CNN Design with ResNet, EfficientNet & ArcFace

---

### 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import timm
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

import sys
sys.path.append('..')

# Settings
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch Version: {torch.__version__}")

# Create output directory
Path('../images').mkdir(exist_ok=True)
Path('../models').mkdir(exist_ok=True)

# Fix random seed
torch.manual_seed(42)
np.random.seed(42)

### 2. ResNet50 Backbone Model

In [ ]:
class LandmarkResNet(nn.Module):
    """
    Landmark Classifier using ResNet50 backbone.
    
    Architecture:
    - ResNet50 Backbone (pretrained on ImageNet)
    - Global Average Pooling (included in ResNet)
    - Dropout for regularization
    - Classification head
    """
    
    def __init__(self, num_classes, pretrained=True, dropout=0.3):
        super(LandmarkResNet, self).__init__()
        
        # Load pretrained ResNet50
        self.backbone = models.resnet50(pretrained=pretrained)
        
        # Get feature dimension
        in_features = self.backbone.fc.in_features  # 2048
        
        # Replace final layer
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )
        
        self.num_classes = num_classes
    
    def forward(self, x):
        return self.backbone(x)
    
    def get_features(self, x):
        """Extract features before classification layer."""
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)
        
        x = self.backbone.avgpool(x)
        x = torch.flatten(x, 1)
        
        return x

# Test the model
num_classes = 100
model = LandmarkResNet(num_classes=num_classes, pretrained=True)
model = model.to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ LandmarkResNet Created")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

# Test forward pass
x = torch.randn(2, 3, 224, 224).to(device)
output = model(x)
print(f"   Input shape: {x.shape}")
print(f"   Output shape: {output.shape}")

### 3. EfficientNet Model

In [ ]:
class LandmarkEfficientNet(nn.Module):
    """
    Landmark Classifier using EfficientNet backbone.
    
    More efficient than ResNet with similar or better accuracy.
    """
    
    def __init__(self, variant='efficientnet_b0', num_classes=1000, pretrained=True, dropout=0.3):
        super(LandmarkEfficientNet, self).__init__()
        
        # Load pretrained EfficientNet
        if variant == 'efficientnet_b0':
            self.backbone = models.efficientnet_b0(pretrained=pretrained)
        elif variant == 'efficientnet_b3':
            self.backbone = models.efficientnet_b3(pretrained=pretrained)
        else:
            self.backbone = models.efficientnet_b0(pretrained=pretrained)
        
        # Get feature dimension
        in_features = self.backbone.classifier[1].in_features  # 1280 for B0, 1536 for B3
        
        # Replace classifier
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes)
        )
        
        self.num_classes = num_classes
    
    def forward(self, x):
        return self.backbone(x)

# Test EfficientNet
efficientnet_model = LandmarkEfficientNet(variant='efficientnet_b0', num_classes=num_classes)
efficientnet_model = efficientnet_model.to(device)

efficientnet_params = sum(p.numel() for p in efficientnet_model.parameters())
print("✅ LandmarkEfficientNet Created")
print(f"   Total parameters: {efficientnet_params:,}")

# Test
output = efficientnet_model(x)
print(f"   Output shape: {output.shape}")

### 4. Vision Transformer (ViT) Model

In [ ]:
def create_vit_model(num_classes, model_name='vit_base_patch16_224', pretrained=True):
    """
    Create Vision Transformer model using TIMM library.
    
    Args:
        num_classes: Number of landmark classes
        model_name: ViT variant (vit_small_patch16_224, vit_base_patch16_224, etc.)
        pretrained: Use pretrained weights
    
    Returns:
        ViT model
    """
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    return model

# Create ViT model
vit_model = create_vit_model(num_classes=num_classes, model_name='vit_small_patch16_224')
vit_model = vit_model.to(device)

vit_params = sum(p.numel() for p in vit_model.parameters())
print("✅ Vision Transformer (ViT-Small) Created")
print(f"   Total parameters: {vit_params:,}")

# Test ViT
output = vit_model(x)
print(f"   Output shape: {output.shape}")

### 5. ArcFace Embedding Model (Metric Learning)

In [ ]:
class ArcFaceHead(nn.Module):
    """
    ArcFace Head for metric learning.
    
    Paper: ArcFace: Additive Angular Margin Loss for Deep Face Recognition
    
    ArcFace adds an angular margin penalty to the softmax loss
    to enhance intra-class compactness and inter-class separability.
    """
    
    def __init__(self, embedding_dim, num_classes, s=30.0, m=0.5):
        super(ArcFaceHead, self).__init__()
        self.s = s  # Scale factor
        self.m = m  # Angular margin
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        
        self.cos_m = np.cos(m)
        self.sin_m = np.sin(m)
        self.th = np.cos(np.pi - m)
        self.mm = np.sin(np.pi - m) * m
    
    def forward(self, embeddings, labels):
        # Normalize
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight))
        
        # Calculate cos(theta + m)
        sine = torch.sqrt(1.0 - torch.clamp(cosine.pow(2), 0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
        # One-hot encoding
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        
        # Apply margin
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        
        return output


class LandmarkEmbeddingModel(nn.Module):
    """
    Landmark model with ArcFace for embedding learning.
    
    Produces normalized embeddings suitable for retrieval.
    """
    
    def __init__(self, num_classes, embedding_dim=512, backbone='resnet50', pretrained=True):
        super(LandmarkEmbeddingModel, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.num_classes = num_classes
        
        # Backbone
        if backbone == 'resnet50':
            base_model = models.resnet50(pretrained=pretrained)
            self.backbone = nn.Sequential(*list(base_model.children())[:-1])
            feature_dim = 2048
        elif backbone == 'efficientnet_b0':
            base_model = models.efficientnet_b0(pretrained=pretrained)
            self.backbone = nn.Sequential(base_model.features, base_model.avgpool)
            feature_dim = 1280
        
        # Embedding layer
        self.embedding = nn.Sequential(
            nn.Linear(feature_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )
        
        # ArcFace head
        self.arcface = ArcFaceHead(embedding_dim, num_classes)
    
    def forward(self, x, labels=None, mode='train'):
        # Extract features
        features = self.backbone(x)
        features = features.view(features.size(0), -1)
        
        # Embeddings
        embeddings = self.embedding(features)
        
        if mode == 'train' and labels is not None:
            logits = self.arcface(embeddings, labels)
            return logits, embeddings
        else:
            # Return normalized embeddings for retrieval
            return F.normalize(embeddings, p=2, dim=1)

# Create ArcFace model
arcface_model = LandmarkEmbeddingModel(
    num_classes=num_classes,
    embedding_dim=512,
    backbone='resnet50'
)
arcface_model = arcface_model.to(device)

arcface_params = sum(p.numel() for p in arcface_model.parameters())
print("✅ LandmarkEmbeddingModel (ArcFace) Created")
print(f"   Total parameters: {arcface_params:,}")

# Test forward passes
logits, embeddings = arcface_model(x, labels=torch.randint(0, num_classes, (2,)), mode='train')
print(f"   Logits shape (ArcFace): {logits.shape}")
print(f"   Embeddings shape: {embeddings.shape}")

# Inference mode
embeddings = arcface_model(x, mode='eval')
print(f"   Normalized embeddings shape: {embeddings.shape}")

### 6. Model Comparison

In [ ]:
# Compare models
models_info = [
    {'name': 'ResNet50', 'params': total_params, 'accuracy': 0.76, 'flops': 4.1},
    {'name': 'EfficientNet-B0', 'params': efficientnet_params, 'accuracy': 0.77, 'flops': 0.39},
    {'name': 'ViT-Small', 'params': vit_params, 'accuracy': 0.79, 'flops': 2.6},
    {'name': 'ArcFace-ResNet50', 'params': arcface_params, 'accuracy': 0.80, 'flops': 4.2},
]

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = [m['name'] for m in models_info]
param_counts = [m['params'] / 1e6 for m in models_info]  # In millions

# Parameters comparison
bars = axes[0].bar(model_names, param_counts, color=['steelblue', 'coral', 'green', 'purple'])
axes[0].set_ylabel('Parameters (Millions)', fontsize=12)
axes[0].set_title('Model Size Comparison', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

for bar, params in zip(bars, param_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                f'{params:.1f}M', ha='center', va='bottom', fontsize=10)

# FLOPs comparison (normalized)
flops = [m['flops'] for m in models_info]
bars2 = axes[1].bar(model_names, flops, color=['steelblue', 'coral', 'green', 'purple'])
axes[1].set_ylabel('FLOPs (Billions)', fontsize=12)
axes[1].set_title('Computational Complexity', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

for bar, f in zip(bars2, flops):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                f'{f:.2f}B', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../images/06_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Model comparison saved to images/06_model_comparison.png")

### 7. Architecture Visualization

In [ ]:
# Visualize model architecture
fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.axis('off')

# Draw architecture boxes
def draw_box(ax, x, y, w, h, text, color='lightblue'):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=10, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Input
draw_box(ax, 0.35, 8.5, 0.3, 0.5, 'Input\n(3×224×224)', 'lightgreen')

# Backbone stages
draw_box(ax, 0.35, 7.5, 0.3, 0.7, 'Conv1\n(64, 112×112)', 'lightyellow')
draw_arrow(ax, 0.5, 8.5, 0.5, 8.2)

draw_box(ax, 0.35, 6.5, 0.3, 0.7, 'ResBlock1\n(256, 56×56)', 'lightyellow')
draw_arrow(ax, 0.5, 7.5, 0.5, 7.2)

draw_box(ax, 0.35, 5.5, 0.3, 0.7, 'ResBlock2\n(512, 28×28)', 'lightyellow')
draw_arrow(ax, 0.5, 6.5, 0.5, 6.2)

draw_box(ax, 0.35, 4.5, 0.3, 0.7, 'ResBlock3\n(1024, 14×14)', 'lightyellow')
draw_arrow(ax, 0.5, 5.5, 0.5, 5.2)

draw_box(ax, 0.35, 3.5, 0.3, 0.7, 'ResBlock4\n(2048, 7×7)', 'lightyellow')
draw_arrow(ax, 0.5, 4.5, 0.5, 4.2)

draw_box(ax, 0.35, 2.5, 0.3, 0.7, 'Global AvgPool\n(2048, 1×1)', 'lightgray')
draw_arrow(ax, 0.5, 3.5, 0.5, 3.2)

draw_box(ax, 0.35, 1.5, 0.3, 0.7, 'Embedding\n(512)', 'lightblue')
draw_arrow(ax, 0.5, 2.5, 0.5, 2.2)

draw_box(ax, 0.35, 0.5, 0.3, 0.7, 'Output\n(N Classes)', 'lightcoral')
draw_arrow(ax, 0.5, 1.5, 0.5, 1.2)

ax.set_xlim(0, 1)
ax.set_ylim(0, 9.5)
ax.set_title('Landmark Detection Model Architecture', fontsize=16, fontweight='bold')
ax.text(0.5, 9.7, 'ResNet50 Backbone → Embedding Layer → Classification Head', 
         ha='center', fontsize=12, style='italic')

plt.tight_layout()
plt.savefig('../images/07_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Architecture diagram saved to images/07_architecture.png")

### 8. Summary

In [ ]:
print("""
📋 MODEL ARCHITECTURE SUMMARY:

1. RESNET50 BACKBONE
   - Pretrained on ImageNet (1.2M images, 1000 classes)
   - Transfer learning for landmark features
   - 2048-dimensional feature vector

2. EFFICIENTNET BACKBONE
   - More parameter efficient
   - Similar accuracy with less computation

3. VISION TRANSFORMER (VIT)
   - State-of-the-art for image classification
   - Attention-based architecture
   - Works well with large-scale datasets

4. ARCFACE EMBEDDING MODEL
   - Metric learning approach
   - Produces normalized embeddings
   - Better for retrieval tasks
   - Angular margin enhances class separation

📚 REFERENCES:
   - ResNet: He et al., "Deep Residual Learning for Image Recognition"
   - EfficientNet: Tan et al., "EfficientNet: Rethinking Model Scaling"
   - ArcFace: Deng et al., "ArcFace: Additive Angular Margin Loss"

🚀 NEXT: Proceed to 04_Training for model training
""")

# Print model summary
print("\n🏗️ MODEL SUMMARY:")
print("-" * 50)
for name, model in [('ResNet50', model), ('EfficientNet-B0', efficientnet_model), 
                     ('ViT-Small', vit_model), ('ArcFace-ResNet50', arcface_model)]:
    params = sum(p.numel() for p in model.parameters())
    print(f"   {name}: {params:,} parameters")